In [ ]:
try:
    import pyspark.sql.functions as F
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

# caminho de entrada dos dados no databricks
dbutils.widgets.text("input_base_path", "/Volumes/workspace/default/inputs", "Pasta de entrada")
input_base_path = dbutils.widgets.get("input_base_path").rstrip("/")
if not input_base_path:
    raise ValueError("Informe o caminho da pasta de entrada no widget do Databricks.")

# lista os arquivos e os nomes das tabelas bronze.
csv_sources = {
    "tb_movies_info": "movies_info_TMDB_IMDB.csv",
    "tb_movies_financials": "movies_financials_IMDB_TMDB.csv",
    "tb_movies_metrics": "movies_metrics_IMDB_TMDB.csv",
    "tb_credits_and_tags": "credits_and_tags_IMDB_TMDB.csv",
    "tb_movies_reviews": "movies_reviews.csv",
}

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [ ]:
# mantém as colunas como texto
csv_read_options = {
    "header": "true",
    "inferSchema": "false",
    "quote": '"',
    "escape": '"',
    "mode": "PERMISSIVE",
}

ingestion_timestamp = F.current_timestamp()

for table_name, file_name in csv_sources.items():
    source_path = f"{input_base_path}/{file_name}"

    (
        spark.read
        .options(**csv_read_options)
        .csv(source_path)
        # adição do horário da carga
        .withColumn("ingestion_datetime", ingestion_timestamp)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(f"bronze.{table_name}")
    )

print(f"Carga concluída: {len(csv_sources)} tabelas foram gravadas na camada Bronze.")

In [ ]:
# verifica o formato das tabelas e a coluna de horário de ingestão dos dados
validation_dfs = []
for table_name in csv_sources:
    table_df = spark.table(f"bronze.{table_name}")
    detail_df = (
        spark.sql(f"DESCRIBE DETAIL bronze.{table_name}")
        .select(
            F.lit(table_name).alias("table_name"),
            F.lit("ingestion_datetime" in table_df.columns).alias("has_ingestion_datetime"),
            F.col("format").alias("storage_format"),
        )
    )
    validation_dfs.append(detail_df)

validation_result = validation_dfs[0]
for validation_df in validation_dfs[1:]:
    validation_result = validation_result.unionByName(validation_df)

display(validation_result)